In [ ]:
!pip install recbole ray kmeans-pytorch "numpy<2.0"

In [3]:
from recbole.quick_start import run_recbole
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import BPR
from recbole.trainer import Trainer

import pandas as pd
import torch

In [11]:
BASE_PATH = "/home/jupyter/project"

config_dict = {
    'model': 'BPR',
    
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'TIME_FIELD': 'timestamp',
    
    
    'load_col': {
        'inter': ['user_id', 'item_id', 'timestamp']
    },

    # глобальное временное разбиение
    'eval_args': {
        'split': {'LS': 'valid_and_test'},
        'order': 'TO', 
        'mode': 'full'
    },


    'epochs': 20,
    'train_batch_size': 2048,
    'learning_rate': 0.001,

    'device': 'cuda',

    'metrics': ['Recall', 'NDCG'],
    'topk': [10],
    'valid_metric': 'NDCG@10',

    'show_progress': True,
    'log_level': 'INFO',


    'checkpoint_dir': f"{BASE_PATH}/saved"
}

In [12]:
def train_bpr(dataset_name, data_path):
    print(f"\n===== Обучение на датасете {dataset_name} =====")
    
    config = Config(
        model='BPR',
        dataset=dataset_name,
        config_dict=config_dict
    )
    
    config['data_path'] = data_path
    
    dataset = create_dataset(config)
    print(dataset)
    
    train_data, valid_data, test_data = data_preparation(config, dataset)
    

    model = BPR(config, train_data.dataset).to(config['device'])
    
    trainer = Trainer(config, model)
    

    best_valid_score, best_valid_result = trainer.fit(
        train_data,
        valid_data,
        saved=True
    )
    
    print("Лучшие результаты на валидации:", best_valid_result)
    
    test_result = trainer.evaluate(test_data)
    print("Итоговое качество модели:", test_result)
    
    return trainer, model, dataset, test_data

In [13]:
datasets = {
    "organic": "/home/jupyter/project/yambda_organic",
    "algo": "/home/jupyter/project/yambda_rec",
    "random": "/home/jupyter/project/yambda_random"
}

In [14]:
trainer_org, model_org, dataset_org, test_org = train_bpr(
    "yambda_organic",
    datasets["organic"]
)


command line args [--shell 12501 --control 12504 --stdin 12503 --hb 12500 --iopub 12502 --debug -f /dev/shm/kernel-connection-file.json] will not be used in RecBole



===== Обучение на датасете yambda_organic =====
yambda_organic
The number of users: 9603
Average actions of users: 1518.903145178088
The number of items: 495187
Average actions of items: 29.452585493127835
The number of inters: 14584508
The sparsity of the dataset: 99.69329869816282%
Remain Fields: ['user_id', 'item_id', 'timestamp']
Лучшие результаты на валидации: OrderedDict([('recall@10', 0.0586), ('ndcg@10', 0.0369)])
Итоговое качество модели: OrderedDict([('recall@10', 0.0536), ('ndcg@10', 0.0323)])


In [16]:
trainer_algo, model_algo, dataset_algo, test_algo = train_bpr(
    "yambda_algo",
    datasets["algo"]
)


command line args [--shell 12501 --control 12504 --stdin 12503 --hb 12500 --iopub 12502 --debug -f /dev/shm/kernel-connection-file.json] will not be used in RecBole



===== Обучение на датасете yambda_algo =====
yambda_algo
The number of users: 8611
Average actions of users: 1693.9033681765388
The number of items: 403197
Average actions of items: 36.17225369299299
The number of inters: 14584508
The sparsity of the dataset: 99.57993073999077%
Remain Fields: ['user_id', 'item_id', 'timestamp']
Лучшие результаты на валидации: OrderedDict([('recall@10', 0.042), ('ndcg@10', 0.0232)])
Итоговое качество модели: OrderedDict([('recall@10', 0.04), ('ndcg@10', 0.0223)])


In [17]:
trainer_rand, model_rand, dataset_rand, test_rand = train_bpr(
    "yambda_random",
    datasets["random"]
)

#после обучения переименовать .pth файл в BPR_random.pth 

command line args [--shell 12501 --control 12504 --stdin 12503 --hb 12500 --iopub 12502 --debug -f /dev/shm/kernel-connection-file.json] will not be used in RecBole



===== Обучение на датасете yambda_random =====
yambda_random
The number of users: 9956
Average actions of users: 1465.0434957307884
The number of items: 507131
Average actions of items: 28.75891388795772
The number of inters: 14584508
The sparsity of the dataset: 99.71114044617401%
Remain Fields: ['user_id', 'item_id', 'timestamp']
Лучшие результаты на валидации: OrderedDict([('recall@10', 0.0445), ('ndcg@10', 0.0258)])
Итоговое качество модели: OrderedDict([('recall@10', 0.0383), ('ndcg@10', 0.0215)])


In [21]:
results = []

def save_result(name, result):
    results.append({
        "dataset": name,
        "recall@10": result['recall@10'],
        "ndcg@10": result['ndcg@10']
    })

save_result("organic", trainer_org.evaluate(test_org, load_best_model=False))
save_result("algo", trainer_algo.evaluate(test_algo, load_best_model=False))
save_result("random", trainer_rand.evaluate(test_rand, load_best_model=False))

import pandas as pd

df = pd.DataFrame(results)
df.to_csv("/home/jupyter/project/results/bpr_results.csv", index=False)

df

,dataset,recall@10,ndcg@10
0,organic,0.0536,0.0323
1,algo,0.0400,0.0223
2,random,0.0383,0.0215


In [31]:
import torch
import numpy as np
from tqdm import tqdm

from recbole.data.interaction import Interaction

def get_topk_predictions(trainer, test_data, k=10):
    model = trainer.model
    model.eval()

    dataset = test_data.dataset
    uid_field = dataset.uid_field
    iid_field = dataset.iid_field

    num_items = dataset.item_num
    num_users = dataset.user_num

    topk_items = []

    for uid in tqdm(range(num_users)):
        interaction = Interaction({
            uid_field: torch.tensor([uid]).to(trainer.device)
        })

        scores = model.full_sort_predict(interaction)
        scores = scores.view(-1)

        # убираем PAD item
        scores[0] = -float('inf')

        _, topk = torch.topk(scores, k)
        topk_items.append(topk.cpu().numpy())

    return np.array(topk_items)

In [32]:
def compute_coverage(topk_items, num_items):
    unique_items = np.unique(topk_items)
    coverage = len(unique_items) / num_items
    return coverage

In [37]:
def compute_novelty(topk_items, dataset):
    item_counts = {}

    inter_feat = dataset.inter_feat
    iid_field = dataset.iid_field

    items = inter_feat[iid_field].numpy()

    for item in items:
        item_counts[item] = item_counts.get(item, 0) + 1

    total_interactions = len(items)

    item_prob = {
        item: count / total_interactions
        for item, count in item_counts.items()
    }

    novelty_scores = []

    for user_items in topk_items:
        user_novelty = 0
        for item in user_items:
            p = item_prob.get(item, 1e-10)
            user_novelty += -np.log2(p)
        novelty_scores.append(user_novelty / len(user_items))

    return np.mean(novelty_scores)

In [38]:
train_org = test_org.dataset
train_algo = test_algo.dataset
train_rand = test_rand.dataset

In [39]:
def evaluate_extra_metrics(name, trainer, train_data, test_data, k=10):
    print(f"\n=== {name} ===")

    topk_items = get_topk_predictions(trainer, test_data, k)

    coverage = compute_coverage(topk_items, test_data.dataset.item_num)
    novelty = compute_novelty(topk_items, train_data)

    print(f"Coverage@{k}: {coverage:.4f}")
    print(f"Novelty@{k}: {novelty:.4f}")

    return {
        "dataset": name,
        "coverage@10": coverage,
        "novelty@10": novelty
    }

In [40]:
extra_results = []

extra_results.append(
    evaluate_extra_metrics("organic", trainer_org, train_org, test_org)
)

extra_results.append(
    evaluate_extra_metrics("algo", trainer_algo, train_algo, test_algo)
)

extra_results.append(
    evaluate_extra_metrics("random", trainer_rand, train_rand, test_rand)
)


=== organic ===


100%|██████████| 9603/9603 [00:05<00:00, 1678.01it/s]


Coverage@10: 0.0318
Novelty@10: 18.4944

=== algo ===


100%|██████████| 8611/8611 [00:04<00:00, 1779.01it/s]


Coverage@10: 0.0224
Novelty@10: 18.6718

=== random ===


100%|██████████| 9956/9956 [00:06<00:00, 1651.67it/s]


Coverage@10: 0.0240
Novelty@10: 19.5380


In [41]:
df.to_csv("/home/jupyter/project/results/bpr_results.csv", index=False)
df_extra.to_csv("/home/jupyter/project/results/bpr_extra_metrics.csv", index=False)

NameError: name 'df_extra' is not defined

In [42]:
extra_results


[{'dataset': 'organic',
  'coverage@10': 0.03182636054662178,
  'novelty@10': 18.494360045441052},
 {'dataset': 'algo',
  'coverage@10': 0.022353836958112286,
  'novelty@10': 18.671785827415636},
 {'dataset': 'random',
  'coverage@10': 0.024003659803877104,
  'novelty@10': 19.538009827307246}]

In [43]:
import pandas as pd

df_extra = pd.DataFrame(extra_results)

In [44]:
df_extra.to_csv("/home/jupyter/project/results/bpr_extra_metrics.csv", index=False)

In [45]:
final_df = df.merge(df_extra, on="dataset")
final_df.to_csv("/home/jupyter/project/results/final_results.csv", index=False)
final_df

,dataset,recall@10,ndcg@10,coverage@10,novelty@10
0,organic,0.0536,0.0323,0.031826,18.494360
1,algo,0.0400,0.0223,0.022354,18.671786
2,random,0.0383,0.0215,0.024004,19.538010


In [48]:
np.save("/home/jupyter/project/results/topk_organic.npy", topk_items)

In [47]:
topk_items = get_topk_predictions(trainer_org, test_org, k=10)

np.save("/home/jupyter/project/results/topk_organic.npy", topk_items)

100%|██████████| 9603/9603 [00:05<00:00, 1676.93it/s]
